# Error Handling and Logging for Data Engineering


## Why Error Handling Matters

In real systems, things ALWAYS fail:

- API goes down
- File missing
- Bad data
- Network issues

If your pipeline crashes → business impact.

👉 A good Data Engineer plans for failure.


## Basic Error Handling


In [ ]:
try:
    x = 10 / 0
except ZeroDivisionError:
    print("Cannot divide by zero")


## Multiple Exceptions


In [ ]:
try:
    data = int("abc")
except ValueError:
    print("Invalid number")
except Exception as e:
    print("Something went wrong:", e)


## Real Pipeline Example


In [ ]:
import requests

def fetch_data(url):
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        return response.json()

    except requests.exceptions.Timeout:
        print("Request timed out")
    except requests.exceptions.HTTPError as e:
        print("HTTP error:", e)
    except Exception as e:
        print("Unexpected error:", e)

    return []


## Why Logging Instead of Print?

`print()`:
- Temporary
- Not structured

`logging`:
- Levels (INFO, ERROR, WARNING)
- Persistent
- Production-ready


_Note:_ In Jupyter, `logging.basicConfig` only applies the **first** time in a session. If levels look wrong, use **Kernel → Restart** and run from this section again—or configure handlers explicitly in production code.


## Basic Logging


In [ ]:
import logging

logging.basicConfig(level=logging.INFO)

logging.info("Pipeline started")
logging.warning("Something looks off")
logging.error("Something failed")


## Logging to File


In [ ]:
logging.basicConfig(
    filename="pipeline.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

logging.info("Pipeline started")


## Production Pattern

**Run the Basic Logging cell first** so `INFO` lines are visible (default root logger level is `WARNING`).


In [ ]:
def run_pipeline(url):
    logging.info("Starting pipeline")

    data = fetch_data(url)

    if not data:
        logging.error("No data fetched")
        return

    logging.info(f"Fetched {len(data)} records")


demo_url = "https://jsonplaceholder.typicode.com/posts"
run_pipeline(demo_url)


## Retry Logic (IMPORTANT)


In [ ]:
import time

def fetch_with_retry(url, retries=3):
    for attempt in range(retries):
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            return response.json()

        except Exception as e:
            print(f"Attempt {attempt+1} failed:", e)
            time.sleep(2)

    return []


## Practice

1. Add logging to your API pipeline
2. Add retry logic
3. Log failures to file


## Assignment

Upgrade your ingestion pipeline:

- Add:
  - Logging to file
  - Retry mechanism
  - Error handling for bad JSON

**Bonus:**
- Log execution time


## Interview Questions

1. Difference between logging and print?
2. What is retry logic?
3. How do you handle pipeline failures?
4. What logs would you track in production?


## What you just learned

- How to handle failure **gracefully**
- How to log like **production** systems
- How to make pipelines **reliable**

👉 This is what companies actually care about.


---

**Next:** **Building a reusable pipeline** — combine modules, stop writing one-off scripts, start shipping **systems**.


## Your tasks

- [ ] Run all cells top-to-bottom (restart kernel if logging behavior looks stuck).
- [ ] Open `pipeline.log` after the file-logging cell; confirm timestamps and levels.
- [ ] **Practice:** Wire logging + retries into your API ingest from notebook `03` (or a small `ingest.py`).
- [ ] **Assignment:** File logging + retries + safe JSON parse (`try`/`except json.JSONDecodeError`); **bonus:** log `time.perf_counter()` duration around the fetch.
- [ ] Answer interview questions in writing; list **four** production log fields you’d alert on (latency, error rate, freshness, etc.).
